## Import Libraries

In [2]:
%pip install -U python-dotenv langchain-community langchain-text-splitters langchain-core langchain-classic langchain-ollama requests

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.6.5-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached aiohttp-3.14.3-cp312-cp312-win_amd64.whl.metadata (8.5 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langsmith-0.14.1-py3-none-any.whl.metadata (22 kB)
  Using cached numpy-2.5.3-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached sqlalchemy-2.1.1-cp312-cp312-win_amd64.whl.metadata


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# PDF loading and text splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector database
from langchain_community.vectorstores import FAISS

# Ollama local LLM and embeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings

# RAG components
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain
)
from langchain_classic.chains import create_retrieval_chain

C:\Users\itibr\AppData\Local\Temp\ipykernel_8188\1221769582.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Load All PDFs

In [7]:
%pip install -U pypdf

  Using cached pypdf-6.19.0-py3-none-any.whl.metadata (7.5 kB)
Using cached pypdf-6.19.0-py3-none-any.whl (395 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader

# Project root directory
BASE_DIR = r"F:\DS PROJECT\Cancer AI Analyst\Cancer AI Analyst"

# Medical PDF file paths
pdf_files = [
    os.path.join(BASE_DIR, "data", "unstructured", "Clinical_Guidelines", "Risk Factors for Cancer.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Medical_Books", "Cancer Staging.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Medical_Books", "Cancer_basics.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Medical_Books", "Diagnosis of Cancer.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Research_Papers", "summary-BreastCan.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Research_Papers", "summary-Cancerris.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Research_Papers", "summary-Cancerscr.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Research_Papers", "summary-Cancertre.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "Research_Papers", "summary-lungsCanc.pdf"),
    os.path.join(BASE_DIR, "data", "unstructured", "WHO", "Cancer fact sheet.pdf"),
]

# Check that all PDFs exist
missing_files = [
    pdf for pdf in pdf_files
    if not os.path.exists(pdf)
]

if missing_files:
    print("Missing PDF files:")
    for pdf in missing_files:
        print(pdf)

    raise FileNotFoundError(
        f"{len(missing_files)} PDF file(s) not found. "
        "Please check the file paths."
    )

# Load all PDF documents
documents = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    documents.extend(docs)

    print(
        f"Loaded: {os.path.basename(pdf)} | "
        f"Pages: {len(docs)}"
    )

print("\nTotal PDFs:", len(pdf_files))
print("Total Pages:", len(documents))

Loaded: Risk Factors for Cancer.pdf | Pages: 2
Loaded: Cancer Staging.pdf | Pages: 3
Loaded: Cancer_basics.pdf | Pages: 12
Loaded: Diagnosis of Cancer.pdf | Pages: 8
Loaded: summary-BreastCan.pdf | Pages: 2
Loaded: summary-Cancerris.pdf | Pages: 1
Loaded: summary-Cancerscr.pdf | Pages: 2
Loaded: summary-Cancertre.pdf | Pages: 1
Loaded: summary-lungsCanc.pdf | Pages: 2
Loaded: Cancer fact sheet.pdf | Pages: 7

Total PDFs: 10
Total Pages: 40


## Load Ollama Model

In [3]:
import requests

try:
    response = requests.get("http://localhost:11434/api/tags")
    print("Status:", response.status_code)
    print(response.json())

except Exception as e:
    print("Connection error:", e)

Status: 200
{'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-09-29T17:43:20.0840241+05:30', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16', 'context_length': 2048, 'embedding_length': 768}, 'capabilities': ['embedding']}, {'name': 'llama3.1:8b', 'model': 'llama3.1:8b', 'modified_at': '2026-09-29T17:29:55.4823807+05:30', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096}, 'capabilities': ['completion', 'tools']}, {'name': 'llama3.1:8b-instruct-q2_K', 'model': 'llama3.1:8b-instruct-q2_K

In [4]:
# Local LLM
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.1
)

# Local embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Ollama LLM and embeddings configured successfully!")

Ollama LLM and embeddings configured successfully!


Test LLM

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0.1,
    num_predict=256,
)

response = llm.invoke("Explain cancer in simple terms.")

print(response.content)

Cancer is a group of diseases that occur when abnormal cells grow and multiply out of control in the body. Here's a simplified explanation:

**What happens:**

1. Normally, cells in our body grow, divide, and die in an orderly way. This process is controlled by a system called the "cell cycle."
2. When a cell is damaged or mutated, it can start to grow and divide uncontrollably. This is called "uncontrolled cell growth."
3. As the abnormal cells continue to grow and multiply, they can form a tumor, which is a mass of abnormal cells.
4. Cancer cells can invade surrounding tissues and spread to other parts of the body, a process called "metastasis."

**Why does it happen:**

Cancer can be caused by a combination of genetic and environmental factors, such as:

* Genetic mutations (changes in the DNA code)
* Exposure to carcinogens (cancer-causing substances, like tobacco smoke or certain chemicals)
* Infections (like human papillomavirus, which can cause cervical cancer)
* Lifestyle choic

## Chunking


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Text splitter configured successfully!")

Text splitter configured successfully!


In [15]:
chunks = text_splitter.split_documents(documents)

print("Total documents:", len(documents))
print("Total chunks:", len(chunks))

# Preview the first chunk
print(chunks[0].page_content[:500])

Total documents: 40
Total chunks: 99
Aspects of a person's age, family
history, lifestyle, and environment
can increase the risk of developing
cancer. Learn more from this Did You
Know? video produced by NCI's
Surveillance, Epidemiology, and End
Results (SEER) program.
Risk Factors for Cancer
It is usually not possible to know exactly why one
person develops cancer and another doesn’t.
But research has shown that certain risk factors
may increase a person’s chances of developing
cancer. (There are also factors that are linked to
a 


## Embedding Model

In [7]:
embedding_model = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")


## Create FAISS VECTOR DB


In [16]:
import os
from langchain_community.vectorstores import FAISS

# Folder to save the vector database
VECTOR_DB_PATH = os.path.join(
    BASE_DIR,
    "vectorstore",
    "faiss_index"
)

os.makedirs(
    os.path.dirname(VECTOR_DB_PATH),
    exist_ok=True
)

# Create FAISS vector database
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Save the database locally
vectorstore.save_local(VECTOR_DB_PATH)

print("FAISS vector database created and saved successfully!")


FAISS vector database created and saved successfully!


In [17]:

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

query = "What are the common risk factors for cancer?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Document {i} ---")
    print(doc.page_content[:700])
    print("Metadata:", doc.metadata)


--- Retrieved Document 1 ---
The list below includes the most-studied known or suspected risk factors for cancer.
Although some of these risk factors can be avoided, others—such as growing older—
cannot. Limiting your exposure to avoidable risk factors may lower your risk of developing
certain cancers.
Age
Alcohol
Cancer-Causing Substances
Chronic Inflammation
Diet
Hormones
Immunosuppression
Infectious Agents
Obesity
Radiation
Sunlight
Tobacco
6/10/26, 11:19 AM Risk Factors for Cancer - NCI
https://www.cancer.gov/about-cancer/causes-prevention/risk 2/2
Metadata: {'producer': 'Skia/PDF m148', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36', 'creationdate': '2026-06-10T05:49:51+00:00', 'title': 'Risk Factors for Cancer - NCI', 'moddate': '2026-06-10T05:49:51+00:00', 'source': 'C:\\Users\\prita\\Downloads\\Cancer AI Analyst\\Cancer AI Analyst\\data\\unstructured\\Clinical_Guidelines\\Risk Factors for Cancer.pdf'

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

print("RAG modules imported successfully!")

RAG modules imported successfully!


In [19]:
system_prompt = """
You are Cancer AI Analyst, an AI assistant that provides
educational information about cancer using the supplied
medical documents.

Follow these rules:

1. Answer using the provided context.
2. If the answer is not available in the context,
   clearly state that the documents do not contain
   sufficient information.
3. Do not invent medical facts, statistics, or references.
4. Explain medical concepts in simple, understandable language.
5. Do not provide a definitive diagnosis or prescribe treatment.
6. Encourage users to consult a qualified healthcare
   professional for personal medical decisions.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

print("Medical RAG prompt created successfully!")

Medical RAG prompt created successfully!


In [20]:
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

print("Document chain created successfully!")

Document chain created successfully!


In [21]:
retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("Cancer AI RAG pipeline created successfully!")

Cancer AI RAG pipeline created successfully!


In [22]:
query = "What are the common risk factors for cancer?"

response = retrieval_chain.invoke({
    "input": query
})

print("Question:", query)

print("\nAnswer:")
print(response["answer"])

Question: What are the common risk factors for cancer?

Answer:
According to the provided documents, the common risk factors for cancer include:

1. Age
2. Tobacco use
3. Alcohol consumption
4. Unhealthy diet
5. Physical inactivity
6. Air pollution
7. Cancer-causing substances
8. Chronic inflammation
9. Hormones
10. Immunodeficiency or immunosuppression
11. Infectious agents
12. Obesity
13. Radiation
14. Sunlight exposure

Additionally, a family history of certain cancers can also be a risk factor, as it may indicate an inherited cancer syndrome.

It's worth noting that some of these risk factors can be avoided or reduced, while others, such as growing older, cannot be changed.


In [23]:
print("Sources:")

for doc in response["context"]:
    source = doc.metadata.get("source", "Unknown source")
    page = doc.metadata.get("page", "Unknown page")

    print(f"Source: {os.path.basename(source)} | Page: {page}")

Sources:
Source: Risk Factors for Cancer.pdf | Page: 1
Source: Cancer fact sheet.pdf | Page: 1
Source: Risk Factors for Cancer.pdf | Page: 0
